# LeetCode Problem Scraper
Scrapes all problem names from https://leetcode.com/problemset/ using the LeetCode GraphQL API.

> LeetCode's public GraphQL API is used here — no login required for problem listings.

In [1]:
# Install required libraries
!pip install requests pandas tqdm --quiet

In [2]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import json

In [3]:
# ── Configuration ──────────────────────────────────────────────────────────────
GRAPHQL_URL = "https://leetcode.com/graphql"
PAGE_SIZE   = 100   # max problems per request (LeetCode allows up to 100)
DELAY_SEC   = 0.5   # polite delay between requests

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (compatible; LeetCode-Scraper/1.0)",
    "Referer": "https://leetcode.com/problemset/",
}

In [4]:
# ── GraphQL query ──────────────────────────────────────────────────────────────
QUERY = """
query problemsetQuestionList($categorySlug: String, $limit: Int, $skip: Int, $filters: QuestionListFilterInput) {
  problemsetQuestionList: questionList(
    categorySlug: $categorySlug
    limit: $limit
    skip: $skip
    filters: $filters
  ) {
    total: totalNum
    questions: data {
      frontendQuestionId: questionFrontendId
      title
      titleSlug
      difficulty
      isPaidOnly
      topicTags {
        name
      }
    }
  }
}
"""

def fetch_page(skip: int, limit: int = PAGE_SIZE) -> dict:
    """Fetch one page of problems from LeetCode GraphQL API."""
    payload = {
        "operationName": "problemsetQuestionList",
        "query": QUERY,
        "variables": {
            "categorySlug": "",
            "skip": skip,
            "limit": limit,
            "filters": {}
        }
    }
    response = requests.post(GRAPHQL_URL, json=payload, headers=HEADERS, timeout=15)
    response.raise_for_status()
    return response.json()

In [5]:
# ── Fetch the first page to learn the total count ─────────────────────────────
first_page = fetch_page(skip=0)
total      = first_page["data"]["problemsetQuestionList"]["total"]
print(f"Total problems on LeetCode: {total}")

Total problems on LeetCode: 3865


In [6]:
# ── Scrape all pages ───────────────────────────────────────────────────────────
all_problems = []

# Collect problems from the first page we already fetched
all_problems.extend(first_page["data"]["problemsetQuestionList"]["questions"])

offsets = range(PAGE_SIZE, total, PAGE_SIZE)

for skip in tqdm(offsets, desc="Fetching pages"):
    try:
        data = fetch_page(skip=skip)
        questions = data["data"]["problemsetQuestionList"]["questions"]
        all_problems.extend(questions)
        time.sleep(DELAY_SEC)
    except requests.RequestException as e:
        print(f"\n⚠️  Error at skip={skip}: {e}. Retrying once...")
        time.sleep(2)
        try:
            data = fetch_page(skip=skip)
            all_problems.extend(data["data"]["problemsetQuestionList"]["questions"])
        except Exception as e2:
            print(f"   Failed again: {e2}")

print(f"\n✅ Collected {len(all_problems)} problems")

Fetching pages: 100%|██████████| 38/38 [00:45<00:00,  1.20s/it]


✅ Collected 3865 problems


In [7]:
# ── Build a clean DataFrame ────────────────────────────────────────────────────
rows = []
for q in all_problems:
    rows.append({
        "id":         q.get("frontendQuestionId", ""),
        "title":      q.get("title", ""),
        "slug":       q.get("titleSlug", ""),
        "difficulty": q.get("difficulty", ""),
        "is_paid":    q.get("isPaidOnly", False),
        "tags":       ", ".join(t["name"] for t in q.get("topicTags", [])),
        "url":        f"https://leetcode.com/problems/{q.get('titleSlug', '')}/"
    })

df = pd.DataFrame(rows)
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df = df.sort_values("id").reset_index(drop=True)

print(df.shape)
df.head(10)

(3865, 7)


,id,title,slug,difficulty,is_paid,tags,url
0,1,Two Sum,two-sum,Easy,False,"Array, Hash Table",https://leetcode.com/problems/two-sum/
1,2,Add Two Numbers,add-two-numbers,Medium,False,"Linked List, Math, Recursion",https://leetcode.com/problems/add-two-numbers/
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,False,"Hash Table, String, Sliding Window",https://leetcode.com/problems/longest-substrin...
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,False,"Array, Binary Search, Divide and Conquer",https://leetcode.com/problems/median-of-two-so...
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,False,"Two Pointers, String, Dynamic Programming",https://leetcode.com/problems/longest-palindro...
5,6,Zigzag Conversion,zigzag-conversion,Medium,False,String,https://leetcode.com/problems/zigzag-conversion/
6,7,Reverse Integer,reverse-integer,Medium,False,Math,https://leetcode.com/problems/reverse-integer/
7,8,String to Integer (atoi),string-to-integer-atoi,Medium,False,String,https://leetcode.com/problems/string-to-intege...
8,9,Palindrome Number,palindrome-number,Easy,False,Math,https://leetcode.com/problems/palindrome-number/
9,10,Regular Expression Matching,regular-expression-matching,Hard,False,"String, Dynamic Programming, Recursion",https://leetcode.com/problems/regular-expressi...


In [8]:
# ── Quick stats ────────────────────────────────────────────────────────────────
print("Difficulty breakdown:")
print(df["difficulty"].value_counts())

print("\nPaid-only vs free:")
print(df["is_paid"].value_counts().rename({True: "Paid", False: "Free"}))

Difficulty breakdown:
difficulty
Medium    2022
Easy       930
Hard       913
Name: count, dtype: int64

Paid-only vs free:
is_paid
Free    3107
Paid     758
Name: count, dtype: int64


In [9]:
# ── Save to CSV ────────────────────────────────────────────────────────────────
output_file = "leetcode_problems.csv"
df.to_csv(output_file, index=False)
print(f"Saved {len(df)} problems to '{output_file}'")

Saved 3865 problems to 'leetcode_problems.csv'


In [10]:
# ── (Optional) Search / filter ─────────────────────────────────────────────────
# Example: show all Easy Array problems
mask = (df["difficulty"] == "Easy") & (df["tags"].str.contains("Array", na=False))
df[mask][["id", "title", "difficulty", "tags"]].head(10)

,id,title,difficulty,tags
0,1,Two Sum,Easy,"Array, Hash Table"
13,14,Longest Common Prefix,Easy,"Array, String, Trie"
25,26,Remove Duplicates from Sorted Array,Easy,"Array, Two Pointers"
26,27,Remove Element,Easy,"Array, Two Pointers"
34,35,Search Insert Position,Easy,"Array, Binary Search"
65,66,Plus One,Easy,"Array, Math"
87,88,Merge Sorted Array,Easy,"Array, Two Pointers, Sorting"
107,108,Convert Sorted Array to Binary Search Tree,Easy,"Array, Divide and Conquer, Tree, Binary Search..."
117,118,Pascal's Triangle,Easy,"Array, Dynamic Programming"
118,119,Pascal's Triangle II,Easy,"Array, Dynamic Programming"
